In [ ]:

import os, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, regularizers


In [ ]:

SEED = 42
N_SPLITS = 5
EPOCHS = 200
BATCH_SIZE = 128
LEARNING_RATE = 1e-3

def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

set_seed(SEED)
print("TensorFlow version:", tf.__version__)


In [ ]:


df = pd.read_csv("https://raw.githubusercontent.com/McahitKutsal/hivcsv/main/HIV7.csv")
print("Dataset shape:", df.shape)
display(df.head())

target_col = "HIV_active"
y = df[target_col].values


X_df = df.drop(columns=[target_col]).select_dtypes(include=["float64", "int64", "float32", "int32"]).copy()
print("Numeric feature shape:", X_df.shape)

X = X_df.values.astype("float32")
n_features = X.shape[1]


In [ ]:

def build_brnn_model(input_length):
    model = models.Sequential([
        layers.Input(shape=(input_length, 1)),
        layers.Bidirectional(
            layers.SimpleRNN(
                64,
                return_sequences=True,
                dropout=0.2,
                recurrent_dropout=0.0,
                kernel_regularizer=regularizers.l2(1e-4)
            )
        ),
        layers.Bidirectional(
            layers.SimpleRNN(
                32,
                return_sequences=False,
                dropout=0.2,
                recurrent_dropout=0.0,
                kernel_regularizer=regularizers.l2(1e-4)
            )
        ),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.2),
        layers.Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
    )
    return model


In [ ]:

def evaluate_predictions(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype("int32")
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob)
    }

def prepare_fold_data(X_train_full, X_test, y_train_full, y_test, random_state):
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full,
        test_size=0.2,
        random_state=random_state,
        stratify=y_train_full
    )

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    X_train = X_train[..., np.newaxis]
    X_val = X_val[..., np.newaxis]
    X_test = X_test[..., np.newaxis]

    return X_train, X_val, X_test, y_train, y_val, y_test


In [ ]:

def run_brnn_5fold_cv(X, y, n_splits=5, seed=42):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    fold_results = []
    histories = []

    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
        print("\n" + "="*28 + f" FOLD {fold} " + "="*28)
        set_seed(seed + fold)

        X_train_full, X_test = X[train_idx], X[test_idx]
        y_train_full, y_test = y[train_idx], y[test_idx]

        X_train, X_val, X_test, y_train, y_val, y_test = prepare_fold_data(
            X_train_full, X_test, y_train_full, y_test, random_state=seed + fold
        )

        model = build_brnn_model(input_length=X_train.shape[1])

        early_stop = callbacks.EarlyStopping(
            monitor="val_loss",
            patience=10,
            restore_best_weights=True,
            verbose=1
        )

        reduce_lr = callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            min_lr=1e-5,
            verbose=1
        )

        history = model.fit(
            X_train, y_train,
            validation_data=(X_val, y_val),
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            callbacks=[early_stop, reduce_lr],
            verbose=0
        )
        histories.append(history.history)

        y_val_prob = model.predict(X_val, verbose=0).ravel()
        y_test_prob = model.predict(X_test, verbose=0).ravel()

        val_metrics = evaluate_predictions(y_val, y_val_prob, threshold=0.5)
        test_metrics = evaluate_predictions(y_test, y_test_prob, threshold=0.5)

        row = {
            "fold": fold,
            "val_accuracy": val_metrics["accuracy"],
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "val_f1": val_metrics["f1"],
            "val_roc_auc": val_metrics["roc_auc"],
            "test_accuracy": test_metrics["accuracy"],
            "test_precision": test_metrics["precision"],
            "test_recall": test_metrics["recall"],
            "test_f1": test_metrics["f1"],
            "test_roc_auc": test_metrics["roc_auc"],
            "epochs_trained": len(history.history["loss"])
        }
        fold_results.append(row)

        print("Validation:",
              {k.replace("val_", ""): round(v, 4) for k, v in row.items() if k.startswith("val_")})
        print("Test:",
              {k.replace("test_", ""): round(v, 4) for k, v in row.items() if k.startswith("test_")})
        print("Epochs trained:", row["epochs_trained"])

    results_df = pd.DataFrame(fold_results)

    metric_cols = ["test_accuracy", "test_precision", "test_recall", "test_f1", "test_roc_auc"]
    summary_rows = []
    for col in metric_cols:
        mean_val = results_df[col].mean()
        std_val = results_df[col].std(ddof=1)
        var_val = results_df[col].var(ddof=1)
        summary_rows.append({
            "Metric": col.replace("test_", "").upper(),
            "Mean": mean_val,
            "Std": std_val,
            "Variance": var_val,
            "Formatted": f"{mean_val:.4f} ± {std_val:.4f}"
        })

    summary_df = pd.DataFrame(summary_rows)
    return results_df, summary_df, histories


In [ ]:

results_df, summary_df, histories = run_brnn_5fold_cv(X, y, n_splits=N_SPLITS, seed=SEED)

print("\nFold bazlı sonuçlar:")
display(results_df)

print("\nÖzet sonuçlar:")
display(summary_df)

results_df.to_csv("BRNN_fold_results.csv", index=False)
summary_df.to_csv("BRNN_summary_results.csv", index=False)

print("Saved: BRNN_fold_results.csv")
print("Saved: BRNN_summary_results.csv")


In [ ]:

def plot_average_history(histories):
    max_len = max(len(h["loss"]) for h in histories)

    train_loss_mat, val_loss_mat = [], []
    train_acc_mat, val_acc_mat = [], []

    for h in histories:
        tl = np.array(h["loss"], dtype=float)
        vl = np.array(h["val_loss"], dtype=float)
        ta = np.array(h["accuracy"], dtype=float)
        va = np.array(h["val_accuracy"], dtype=float)

        def pad(arr, length):
            if len(arr) < length:
                arr = np.pad(arr, (0, length-len(arr)), mode="edge")
            return arr

        train_loss_mat.append(pad(tl, max_len))
        val_loss_mat.append(pad(vl, max_len))
        train_acc_mat.append(pad(ta, max_len))
        val_acc_mat.append(pad(va, max_len))

    avg_train_loss = np.mean(train_loss_mat, axis=0)
    avg_val_loss = np.mean(val_loss_mat, axis=0)
    avg_train_acc = np.mean(train_acc_mat, axis=0)
    avg_val_acc = np.mean(val_acc_mat, axis=0)

    epochs = np.arange(1, max_len + 1)

    plt.figure(figsize=(8,5))
    plt.plot(epochs, avg_train_acc, label="Train Accuracy")
    plt.plot(epochs, avg_val_acc, label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("BRNN Average Accuracy Across 5 Folds")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(8,5))
    plt.plot(epochs, avg_train_loss, label="Train Loss")
    plt.plot(epochs, avg_val_loss, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("BRNN Average Loss Across 5 Folds")
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
plot_average_history(histories)

In [ ]:
def plot_average_history(histories):
    import numpy as np
    import matplotlib.pyplot as plt

    max_len = max(len(h["loss"]) for h in histories)

    train_loss_mat, val_loss_mat = [], []
    train_acc_mat, val_acc_mat = [], []

    for h in histories:
        tl = np.array(h["loss"], dtype=float)
        vl = np.array(h["val_loss"], dtype=float)
        ta = np.array(h["accuracy"], dtype=float)
        va = np.array(h["val_accuracy"], dtype=float)

        def pad(arr, length):
            if len(arr) < length:
                arr = np.pad(arr, (0, length - len(arr)), mode="edge")
            return arr

        train_loss_mat.append(pad(tl, max_len))
        val_loss_mat.append(pad(vl, max_len))
        train_acc_mat.append(pad(ta, max_len))
        val_acc_mat.append(pad(va, max_len))

    avg_train_loss = np.mean(train_loss_mat, axis=0)
    avg_val_loss = np.mean(val_loss_mat, axis=0)
    avg_train_acc = np.mean(train_acc_mat, axis=0)
    avg_val_acc = np.mean(val_acc_mat, axis=0)

    epochs = np.arange(1, max_len + 1)

  
    stop_epochs = [len(h["loss"]) for h in histories]

   
    early_stop_epoch = int(np.percentile(stop_epochs, 75))
    early_stop_epoch = max(1, min(early_stop_epoch, max_len))

  
    early_acc_y = avg_val_acc[early_stop_epoch - 1]
    early_loss_y = avg_val_loss[early_stop_epoch - 1]

    
    plt.figure(figsize=(8,5))
    plt.plot(epochs, avg_train_acc, linewidth=2, label="Train Accuracy")
    plt.plot(epochs, avg_val_acc, linewidth=2, label="Validation Accuracy")

    plt.axvline(x=early_stop_epoch, color="red", linestyle="--", linewidth=2)
    plt.scatter(early_stop_epoch, early_acc_y, color="red", s=80, zorder=5)
    plt.text(
        early_stop_epoch + 1,
        early_acc_y,
        f"Early Stop Epoch = {early_stop_epoch}",
        color="red",
        fontsize=10
    )

    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("BRNN Average Accuracy Across 5 Folds")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    
    plt.figure(figsize=(8,5))
    plt.plot(epochs, avg_train_loss, linewidth=2, label="Train Loss")
    plt.plot(epochs, avg_val_loss, linewidth=2, label="Validation Loss")

    plt.axvline(x=early_stop_epoch, color="red", linestyle="--", linewidth=2)
    plt.scatter(early_stop_epoch, early_loss_y, color="red", s=80, zorder=5)
    plt.text(
        early_stop_epoch + 1,
        early_loss_y,
        f"Early Stop Epoch = {early_stop_epoch}",
        color="red",
        fontsize=10
    )

    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("BRNN Average Loss Across 5 Folds")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("Fold-based stop epochs:", stop_epochs)
    print("75th Percentile Early Stop Epoch:", early_stop_epoch)


In [ ]:
def plot_average_history(histories):
    import numpy as np
    import matplotlib.pyplot as plt

    max_len = max(len(h["loss"]) for h in histories)

    train_loss_mat, val_loss_mat = [], []
    train_acc_mat, val_acc_mat = [], []

    for h in histories:
        tl = np.array(h["loss"], dtype=float)
        vl = np.array(h["val_loss"], dtype=float)
        ta = np.array(h["accuracy"], dtype=float)
        va = np.array(h["val_accuracy"], dtype=float)

        def pad(arr, length):
            if len(arr) < length:
                arr = np.pad(arr, (0, length - len(arr)), mode="edge")
            return arr

        train_loss_mat.append(pad(tl, max_len))
        val_loss_mat.append(pad(vl, max_len))
        train_acc_mat.append(pad(ta, max_len))
        val_acc_mat.append(pad(va, max_len))

    avg_train_loss = np.mean(train_loss_mat, axis=0)
    avg_val_loss = np.mean(val_loss_mat, axis=0)
    avg_train_acc = np.mean(train_acc_mat, axis=0)
    avg_val_acc = np.mean(val_acc_mat, axis=0)

    epochs = np.arange(1, max_len + 1)

    stop_epochs = [len(h["loss"]) for h in histories]
    early_stop_epoch = int(np.percentile(stop_epochs, 80))
    early_stop_epoch = max(1, min(early_stop_epoch, max_len))

    early_acc_y = avg_val_acc[early_stop_epoch - 1]
    early_loss_y = avg_val_loss[early_stop_epoch - 1]

    plt.figure(figsize=(8,5))
    plt.plot(epochs, avg_train_acc, linewidth=2, label="Train Accuracy")
    plt.plot(epochs, avg_val_acc, linewidth=2, label="Validation Accuracy")
    plt.axvline(x=early_stop_epoch, color="red", linestyle="--", linewidth=2)
    plt.scatter(early_stop_epoch, early_acc_y, color="red", s=80, zorder=5)
    plt.text(
        early_stop_epoch + 1,
        early_acc_y,
        f"Early Stop Epoch = {early_stop_epoch}",
        color="red",
        fontsize=10
    )
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("BRNN Average 5 Folds Accuracy Across ")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(8,5))
    plt.plot(epochs, avg_train_loss, linewidth=2, label="Train Loss")
    plt.plot(epochs, avg_val_loss, linewidth=2, label="Validation Loss")
    plt.axvline(x=early_stop_epoch, color="red", linestyle="--", linewidth=2)
    plt.scatter(early_stop_epoch, early_loss_y, color="red", s=80, zorder=5)
    plt.text(
        early_stop_epoch + 1,
        early_loss_y,
        f"Early Stop Epoch = {early_stop_epoch}",
        color="red",
        fontsize=10
    )
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("BRNN Average 5-Folds Loss Across")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    print("Fold-based stop epochs:", stop_epochs)
    print("75th Percentile Early Stop Epoch:", early_stop_epoch)


plot_average_history(histories)